In [ ]:
import sys
sys.path.append('../datasets')
sys.path.append('../algorithms')
sys.path.append('../test')


import torch

import matplotlib.pyplot as plt
from pansharpening_static import PANHandler
from load_pavia import load_pavia
from pan_ctv import PANTVCTV
from torchvision import transforms
from math import sqrt
from PIL import Image , ImageFont,ImageDraw

In [2]:
# Define device (default is "cpu")
device = "cpu" 

    # Define dtype
dtype = torch.float64
crop_size = 256
    # Define random seed
seed = 42
torch.manual_seed(seed)

    # Define data path
data_path = '../../data/PaviaU.mat'

rgb = [29,19,9]

In [ ]:
hsi_data = load_pavia(data_path)
_, b, h,w = hsi_data.shape
PAN = PANHandler(nband=b, 
                 size=(h,w),
                 scale=4,
                 sigma=1,
                 noise_level=0.1,
                 seed=42 
                 )

In [4]:
# Matrice de transformation
X = hsi_data

Y_H = PAN.simulate_low_res_hsi(X)  
Y_M = PAN.get_panchromatic(X)

In [5]:
A,A_adj, R, R_adj = PAN.get_operators()

solver_gp = PANTVCTV(
    A=A,
    Aadj=A_adj,
    spectral_op = R,
    spectral_op_t = R_adj,
    max_iter=50,
    lmbda=0.2,
    lmbda_m=2,
    tol=1e-7,
    scale=PAN.scale,
    p = float('inf'),
    q = 1,
    r = 1,
    verbose=True,
    
)

In [ ]:
U_gp, costs_gp = solver_gp(Y_H, Y_M)

In [ ]:

x_rgb = X[0, rgb,...].cpu().numpy().transpose(1, 2, 0)
x_rgb = (x_rgb - x_rgb.min())/(x_rgb.max() - x_rgb.min())

y_rgb = Y_H[0, rgb, ...].cpu().numpy().transpose(1, 2, 0)
y_rgb = (y_rgb - y_rgb.min())/(y_rgb.max() - y_rgb.min())

z_rgb = U_gp[0, rgb, ...].cpu().numpy().transpose(1, 2, 0)
z_rgb = (z_rgb - z_rgb.min())/(z_rgb.max() - z_rgb.min())

plt.figure(figsize=(15, 5))
plt.subplot(141)
plt.imshow(x_rgb)
# plt.title('RGB')
plt.axis('off')

plt.subplot(142)
plt.imshow(Y_M[0, 0, ...].cpu().numpy(), cmap='gray')
# plt.title('Panchromatic')
plt.axis('off')

plt.subplot(143)
plt.imshow(y_rgb)
# plt.title('Noisy RGB')
plt.axis('off')

plt.subplot(144)
plt.imshow(z_rgb)
# plt.title('Noisy RGB')=4
plt.axis('off')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données d'exemple (à adapter avec vos données réelles)
cout_total = costs_gp

# Création du vecteur d'itérations de même longueur que cout_total
iterations = np.arange(0, len(cout_total))  # Pas de 10 entre les mesures

# Vérification des dimensions
print(f"Dimensions vérifiées: iterations {iterations.shape}, cout_total {len(cout_total)}")

# Création du graphique
plt.figure(figsize=(12, 6))
plt.semilogy(iterations, cout_total, 'b-', linewidth=1.5, label='Coût total')

# Personnalisation
plt.xlabel('Itérations', fontsize=12)
plt.ylabel('Coût total (échelle log10)', fontsize=12)
plt.title('Évolution du Coût Total', fontsize=14)
plt.grid(True, which="both", linestyle='--', alpha=0.6)

# Affichage
plt.legend()
plt.tight_layout()
plt.show()

In [31]:
import torch
from torchvision import transforms
from PIL import Image
to_pil = transforms.ToPILImage()

In [32]:
image1 = to_pil(x_rgb)
image1.save('image1.png')

In [33]:
image2 = to_pil(Y_M[0, 0, ...])
image2.save('image2.png')

In [34]:
image3 = to_pil(y_rgb)
image3.save('image3.png')

In [35]:
image4 = to_pil(z_rgb)
image4.save('image4.png')